# Phase 1 - Baseline training (Colab T4)

Trains both YOLOv8n detectors on their own datasets, 100 epochs, default
hyperparameters. See `context.md` / `PROGRESS.md` for the full plan.

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Add your Roboflow key in Colab **Secrets** (key icon, left sidebar):
   name `ROBOFLOW_API_KEY`, toggle notebook access on.
3. The repo must be pushed to GitHub (this notebook clones it).

The two models are independent. Each trains, then immediately saves its
own artifacts - so a mid-session disconnect after the child run still
leaves you the child weights. You can also run just one model's cells.

In [ ]:
!nvidia-smi

In [ ]:
# Pinned: the sweep + baselines ran on 8.4.106. Ablations must
# compare like with like, and 8.4.x changed head init
# ("Remapped N/12 cls head rows...") vs 8.3.x.
!pip install -q ultralytics==8.4.106 roboflow

In [ ]:
# Clone the repo (or pull if already present) and cd into it
import os
REPO_URL = "https://github.com/FooJames/DEEPLRN_Group2.git"
if not os.path.isdir("DEEPLRN_Group2"):
    !git clone $REPO_URL
else:
    !cd DEEPLRN_Group2 && git pull
%cd DEEPLRN_Group2

In [ ]:
# Roboflow API key from Colab Secrets (never hardcode it)
from google.colab import userdata
import os
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("key loaded:", bool(os.environ.get("ROBOFLOW_API_KEY")))

In [ ]:
# Download both datasets into data/ (versions pinned so the split can't drift)
!python scripts/download_data.py --child-version 3 --hazard-version 1

In [ ]:
# Fix Roboflow's broken yaml paths + collapse child to a single class.
# Both are needed every fresh download (Colab included).
!python scripts/fix_data_yaml.py data/child/data.yaml data/hazard/data.yaml
!python scripts/normalize_child_labels.py data/child

## Child detector - train (100 epochs)

In [ ]:
!python scripts/train.py --model child --data data/child/data.yaml --epochs 100

### Child detector - save artifacts (run right after training)

In [ ]:
# Bundle the child run (weights + plots + confusion matrix) and its metrics,
# then download. Do this before starting the hazard run.
!zip -r child_artifacts.zip runs/detect/child_baseline results/metrics/baseline_child.csv
from google.colab import files
files.download("child_artifacts.zip")

## Hazard detector - train (100 epochs)

In [ ]:
!python scripts/train.py --model hazard --data data/hazard/data.yaml --epochs 100

### Hazard detector - save artifacts (run right after training)

In [ ]:
# Bundle the hazard run (weights + per-class plots) and its metrics, then download.
!zip -r hazard_artifacts.zip runs/detect/hazard_baseline results/metrics/baseline_hazard.csv
from google.colab import files
files.download("hazard_artifacts.zip")

## Baseline mAP (both models)

In [ ]:
!cat results/metrics/baseline_child.csv results/metrics/baseline_hazard.csv

### Notes
- Re-running the download cell overwrites `data/` - re-run the fix + normalize
  cell after any re-download.
- Keep the creators' train/val/test splits; do not reshuffle.
- This is the **baseline** (default hyperparameters). Tuning = Phase 2.
- The test split is for the very end only - don't evaluate on it here.